# RepoScribe — Colab driver

**An agentic documentation generator that cannot hallucinate.**

RepoScribe reads a code package and generates an API reference, a user guide, and a changelog in which *every documented symbol and every citation is verified against a ground-truth symbol table* (extracted with tree-sitter). Anything the model invents is pruned before it is written.

This notebook runs top-to-bottom with **no API key** using a deterministic offline mock LLM — that reproduces the tests and every eval number. The last (optional) cell runs *live* against real Gemini.

**Steps:** setup → (1) offline tests → (2) evaluation + guardrail ablation → (3) generate & render docs → (4) optional live run.

Default target: the real **`@lwc/module-resolver`** package from [salesforce/lwc](https://github.com/salesforce/lwc).

## Setup — locate the code and install dependencies

Upload the `reposcribe/` folder to Colab (Files panel → upload) **or** mount Google Drive so this notebook can see the `src/reposcribe/` package, then run this cell.

In [ ]:
import os, sys, glob, subprocess

def find_repo_root():
    for c in [".", "reposcribe", "/content/reposcribe", "/content"]:
        if os.path.isfile(os.path.join(c, "src", "reposcribe", "__init__.py")):
            return os.path.abspath(c)
    for hit in glob.glob("/content/**/src/reposcribe/__init__.py", recursive=True):
        return os.path.dirname(os.path.dirname(os.path.dirname(hit)))
    return None

ROOT = find_repo_root()
if ROOT is None:
    raise SystemExit(
        "Could not find the RepoScribe code. Upload the `reposcribe/` folder "
        "(Files panel → upload) or mount Google Drive, then re-run this cell."
    )
os.chdir(ROOT)
if os.path.join(ROOT, "src") not in sys.path:
    sys.path.insert(0, os.path.join(ROOT, "src"))
print("RepoScribe root:", ROOT)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Dependencies installed.")

## Step 1 — Tests (offline, deterministic)

The whole pipeline is exercised with the mock LLM. Expect **22 passed**.

In [ ]:
!{sys.executable} -m pytest -q

## Step 2 — Evaluation harness + guardrail ablation (offline)

Scores 14 cases against the ground-truth `SymbolTable` and prints the four metrics, then a fault-injection ablation showing the groundedness guardrail catching a hallucinated symbol and a bogus citation.

In [ ]:
!{sys.executable} eval/run_eval.py

## Step 3 — Generate and render the docs (offline)

Runs the full agentic pipeline on the frozen `@lwc/module-resolver` source and renders the generated API reference inline. `auto_yes=True` skips the human approval gate for the notebook.

In [ ]:
from reposcribe.config import Settings
from reposcribe.pipeline import run
from IPython.display import Markdown, display

settings = Settings.from_env(mock=True)
state = run("eval/fixtures/lwc-module-resolver", "out", settings, auto_yes=True, verbose=True)
print("\nmetrics:", state.metrics)

api = next(a for a in state.artifacts if a.kind == "api_reference")
display(Markdown(api.markdown))

## Step 4 (optional) — Live run against real Gemini

Add your key in **Colab → Secrets** (the 🔑 icon in the left sidebar) as `GEMINI_API_KEY`, enable notebook access, then run this cell. It sparse-clones just the target package from the real LWC repo (falling back to the frozen fixture) and documents it with real Gemini output — the guardrails still verify and prune everything against the symbol table.

In [ ]:
from reposcribe.config import Settings, get_api_key
from reposcribe.pipeline import run
from IPython.display import Markdown, display

assert get_api_key(), "No GEMINI_API_KEY found. Add it in Colab → Secrets, then re-run."

TARGET = "eval/fixtures/lwc-module-resolver"
if not os.path.isdir("lwc"):
    subprocess.run("git clone --depth 1 --filter=blob:none --sparse "
                   "https://github.com/salesforce/lwc.git", shell=True, check=False)
    subprocess.run("cd lwc && git sparse-checkout set packages/@lwc/module-resolver",
                   shell=True, check=False)
cloned = "lwc/packages/@lwc/module-resolver/src"
if os.path.isdir(cloned):
    TARGET = cloned
print("Documenting:", TARGET)

settings = Settings.from_env(mock=False)
state = run(TARGET, "out_live", settings, auto_yes=True, verbose=True)
print("\nmetrics:", state.metrics)
display(Markdown(next(a for a in state.artifacts if a.kind == "api_reference").markdown))

---

Generated docs are written to `out/` (offline) and `out_live/` (live): `api_reference.md`, `user_guide.md`, `changelog.md`, plus `workspace_state.json` (the episodic-memory run trace). See `docs/architecture.md` for the design and `eval/eval_report.md` for the full evaluation.